# Experiment: TravelPlanner 3-Way Comparison on OpenRouter Qwen3.5-9B

This notebook prepares a **same-provider / same-model / same-scorer** comparison between three setups:
- **Solo model**: one direct planning call, no swarm, no multi-agent orchestration
- **StigmergiAgentic V3**: this repository
- **SwarmAgentic**: external framework from [YaoZ720/SwarmAgenticCode](https://github.com/YaoZ720/SwarmAgenticCode/tree/main)

## Controlled dimensions

All three arms are evaluated with:
- provider: **OpenRouter**
- model: **`qwen/qwen3.5-9b`**
- split: **`validation`**
- scorer: **official TravelPlanner scorer** from this repository
- same evaluated query range (`MAX_QUERIES`)

## Important caveat

This notebook controls model/provider/scorer/split, but it does **not** automatically equalize optimization budget between methods. In particular, SwarmAgentic includes a PSO phase before evaluation. So the result is a strong **framework comparison under a shared model**, but not yet a fully cost-matched study.


## Prerequisites

Before running this notebook:
- `OPENROUTER_API_KEY` must be available in the Jupyter kernel environment
- `uv` must be installed
- local repo dependencies must already be usable

All generated artifacts are written under `output/travelplanner_framework_compare/<run_tag>/`.


In [ ]:
from __future__ import annotations

import json
import os
import shlex
import subprocess
from datetime import datetime, timezone
from pathlib import Path
from typing import Any

from IPython.display import Markdown, display

REPO_ROOT = Path.cwd().resolve()
assert (REPO_ROOT / 'main.py').exists(), 'Run this notebook from the repository root.'


def _merge_env(extra: dict[str, str] | None = None) -> dict[str, str]:
    env = dict(os.environ)
    if extra:
        env.update({key: str(value) for key, value in extra.items()})
    return env


def run_command(
    cmd: list[str],
    *,
    cwd: Path | None = None,
    env: dict[str, str] | None = None,
    check: bool = True,
    log_path: Path | None = None,
) -> subprocess.CompletedProcess[str]:
    merged_env = _merge_env(env)
    print('$', shlex.join(cmd))
    proc = subprocess.run(
        cmd,
        cwd=str(cwd) if cwd is not None else None,
        env=merged_env,
        text=True,
        capture_output=True,
        check=False,
    )
    combined = proc.stdout
    if proc.stderr:
        combined += ('\n' if combined else '') + proc.stderr
    if log_path is not None:
        log_path.parent.mkdir(parents=True, exist_ok=True)
        log_path.write_text(combined, encoding='utf-8')
    if combined.strip():
        print(combined.strip()[:8000])
    if check and proc.returncode != 0:
        raise RuntimeError(f'Command failed with exit={proc.returncode}: {shlex.join(cmd)}')
    return proc


def load_json(path: Path, default: Any = None) -> Any:
    if not path.exists():
        return default
    return json.loads(path.read_text(encoding='utf-8'))


def write_json(path: Path, payload: Any) -> None:
    path.parent.mkdir(parents=True, exist_ok=True)
    path.write_text(json.dumps(payload, indent=2, ensure_ascii=False) + '\n', encoding='utf-8')


def extract_last_json(stdout: str) -> dict[str, Any]:
    text = stdout.strip()
    try:
        return json.loads(text)
    except json.JSONDecodeError:
        start = text.find('{')
        end = text.rfind('}')
        if start >= 0 and end > start:
            return json.loads(text[start:end + 1])
        raise


print(REPO_ROOT)


In [ ]:
RUN_TAG = os.environ.get('TRAVELPLANNER_COMPARE_RUN_TAG') or datetime.now(timezone.utc).strftime('%Y%m%d_%H%M%S')
MODEL_NAME = os.environ.get('TRAVELPLANNER_COMPARE_MODEL', 'qwen/qwen3.5-9b')
OPENROUTER_BASE_URL = os.environ.get('TRAVELPLANNER_COMPARE_BASE_URL', 'https://openrouter.ai/api/v1')
SPLIT = os.environ.get('TRAVELPLANNER_COMPARE_SPLIT', 'validation')
MAX_QUERIES = int(os.environ.get('TRAVELPLANNER_COMPARE_MAX_QUERIES', '180'))
SOLO_MAX_BUDGET_USD = float(os.environ.get('TRAVELPLANNER_COMPARE_SOLO_BUDGET_USD', '20'))
OUR_MAX_TICKS = int(os.environ.get('TRAVELPLANNER_COMPARE_OUR_MAX_TICKS', '30'))
OUR_AGENT_COUNT = int(os.environ.get('TRAVELPLANNER_COMPARE_OUR_AGENTS', '3'))
OUR_MAX_BUDGET_USD = float(os.environ.get('TRAVELPLANNER_COMPARE_OUR_BUDGET_USD', '20'))
SWARM_MAX_ITERATION = int(os.environ.get('TRAVELPLANNER_COMPARE_SWARM_MAX_ITERATION', '5'))
SWARM_SAMPLE_STEP = int(os.environ.get('TRAVELPLANNER_COMPARE_SWARM_SAMPLE_STEP', '5'))
SWARM_MAX_WORKERS = int(os.environ.get('TRAVELPLANNER_COMPARE_SWARM_MAX_WORKERS', '8'))
SWARM_EXTRACT_MODEL = os.environ.get('TRAVELPLANNER_COMPARE_SWARM_EXTRACT_MODEL', MODEL_NAME)

RUN_SOLO = os.environ.get('TRAVELPLANNER_COMPARE_RUN_SOLO', '1') == '1'
RUN_OUR = os.environ.get('TRAVELPLANNER_COMPARE_RUN_OUR', '1') == '1'
RUN_SWARM = os.environ.get('TRAVELPLANNER_COMPARE_RUN_SWARM', '1') == '1'
TRAIN_SWARM = os.environ.get('TRAVELPLANNER_COMPARE_TRAIN_SWARM', '1') == '1'
INSTALL_SWARM_DEPS = os.environ.get('TRAVELPLANNER_COMPARE_INSTALL_SWARM_DEPS', '1') == '1'
CLONE_SWARM = os.environ.get('TRAVELPLANNER_COMPARE_CLONE_SWARM', '1') == '1'

COMPARE_ROOT = REPO_ROOT / 'output' / 'travelplanner_framework_compare' / RUN_TAG
SOLO_ROOT = COMPARE_ROOT / 'solo'
OUR_ROOT = COMPARE_ROOT / 'stigmergiagentic'
SWARM_ROOT = COMPARE_ROOT / 'swarmagentic'
SWARM_CLONE = SWARM_ROOT / 'repo'
TABLE_ROOT = COMPARE_ROOT / 'comparison'

SOLO_RUNS_JSON = SOLO_ROOT / 'runs.json'
SOLO_OFFICIAL_JSON = SOLO_ROOT / 'official_eval.json'
SOLO_CONFIG_PATH = SOLO_ROOT / 'config_qwen35_9b_openrouter.yaml'
OUR_RUNS_JSON = OUR_ROOT / 'runs.json'
OUR_OFFICIAL_JSON = OUR_ROOT / 'official_eval.json'
OUR_CONFIG_PATH = OUR_ROOT / 'config_qwen35_9b_openrouter.yaml'
SWARM_RESULTS_JSONL = SWARM_ROOT / 'evaluation' / 'test' / 'results.jsonl'
SWARM_RUNS_JSON = SWARM_ROOT / 'runs.json'
SWARM_OFFICIAL_JSON = SWARM_ROOT / 'official_eval.json'
TABLE_MD = TABLE_ROOT / 'comparison_table.md'
TABLE_CSV = TABLE_ROOT / 'comparison_table.csv'
TABLE_JSON = TABLE_ROOT / 'comparison_table.json'
NOTEBOOK_SUMMARY = COMPARE_ROOT / 'notebook_summary.json'

for directory in [COMPARE_ROOT, SOLO_ROOT, OUR_ROOT, SWARM_ROOT, TABLE_ROOT]:
    directory.mkdir(parents=True, exist_ok=True)

summary = {
    'run_tag': RUN_TAG,
    'model_name': MODEL_NAME,
    'split': SPLIT,
    'max_queries': MAX_QUERIES,
    'our_max_ticks': OUR_MAX_TICKS,
    'our_agents': OUR_AGENT_COUNT,
    'swarm_max_iteration': SWARM_MAX_ITERATION,
    'swarm_sample_step': SWARM_SAMPLE_STEP,
    'swarm_max_workers': SWARM_MAX_WORKERS,
    'paths': {
        'compare_root': str(COMPARE_ROOT),
        'solo_official_json': str(SOLO_OFFICIAL_JSON),
        'our_official_json': str(OUR_OFFICIAL_JSON),
        'swarm_official_json': str(SWARM_OFFICIAL_JSON),
        'table_md': str(TABLE_MD),
    },
}
summary


In [ ]:
if not os.environ.get('OPENROUTER_API_KEY'):
    raise EnvironmentError('OPENROUTER_API_KEY is missing in the notebook environment.')

print('OPENROUTER_API_KEY detected')
print(json.dumps(summary, indent=2))


In [ ]:
run_command(['uv', 'run', 'python', 'scripts/setup_travelplanner.py'], cwd=REPO_ROOT, check=True, log_path=COMPARE_ROOT / 'setup_data.log')

count_proc = run_command(
    ['uv', 'run', 'python', '-c', "from datasets import load_dataset; ds = load_dataset('osunlp/TravelPlanner', 'validation'); print(len(ds['validation']))"],
    cwd=REPO_ROOT,
    check=True,
    log_path=COMPARE_ROOT / 'dataset_count.log',
)
print('validation_count=', count_proc.stdout.strip().splitlines()[-1])


## Shared config overrides

The next cell writes two override config files:
- one for the solo baseline
- one for StigmergiAgentic V3


In [ ]:
solo_config_text = f"""
travelplanner:
  dataset_split: "{SPLIT}"

llm:
  provider: "openrouter"
  model: "{MODEL_NAME}"
  max_budget_usd: {SOLO_MAX_BUDGET_USD}
  max_tokens_total: 2000000
  request_timeout_seconds: 300
  retry_attempts: 3
""".strip() + '\n'
SOLO_CONFIG_PATH.write_text(solo_config_text, encoding='utf-8')

our_config_text = f"""
travelplanner:
  dataset_split: "{SPLIT}"

agents:
  num_agents: {OUR_AGENT_COUNT}

orchestrator:
  max_ticks: {OUR_MAX_TICKS}

llm:
  provider: "openrouter"
  model: "{MODEL_NAME}"
  max_budget_usd: {OUR_MAX_BUDGET_USD}
  max_tokens_total: 4000000
  request_timeout_seconds: 300
  retry_attempts: 3
""".strip() + '\n'
OUR_CONFIG_PATH.write_text(our_config_text, encoding='utf-8')

print(SOLO_CONFIG_PATH)
print(OUR_CONFIG_PATH)


## Arm 1: Solo model baseline

This arm uses a single direct planning call through [run_travelplanner_solo_query_export.py](/Users/lotfi/Documents/EMLV/Memoire/StigmergiAgentic/scripts/run_travelplanner_solo_query_export.py), then applies the same official scorer.


In [ ]:
def run_solo_validation() -> dict[str, Any]:
    runs: list[dict[str, Any]] = []
    query_dir = SOLO_ROOT / 'queries'
    query_dir.mkdir(parents=True, exist_ok=True)

    for query_idx in range(MAX_QUERIES):
        cmd = [
            'uv', 'run', 'python', 'scripts/run_travelplanner_solo_query_export.py',
            '--objective', f'Query {query_idx}',
            '--query-idx', str(query_idx),
            '--config', str(SOLO_CONFIG_PATH),
        ]
        proc = run_command(
            cmd,
            cwd=REPO_ROOT,
            check=True,
            log_path=query_dir / f'query_{query_idx:03d}.log',
        )
        payload = extract_last_json(proc.stdout)
        (query_dir / f'query_{query_idx:03d}.json').write_text(
            json.dumps(payload, indent=2, ensure_ascii=False) + '\n',
            encoding='utf-8',
        )
        runs.append(payload)

    write_json(SOLO_RUNS_JSON, {'runs': runs})
    run_command(
        [
            'uv', 'run', 'python', 'scripts/eval_travelplanner_official.py',
            '--runs-json', str(SOLO_RUNS_JSON),
            '--database-root', 'data/travelplanner/database',
            '--split', SPLIT,
            '--out', str(SOLO_OFFICIAL_JSON),
        ],
        cwd=REPO_ROOT,
        check=True,
        log_path=SOLO_ROOT / 'official_eval.log',
    )
    official_payload = load_json(SOLO_OFFICIAL_JSON, {})
    return official_payload.get('scores', {}) if isinstance(official_payload, dict) else {}


solo_scores = None
if RUN_SOLO:
    solo_scores = run_solo_validation()
solo_scores


## Arm 2: StigmergiAgentic V3


In [ ]:
def run_stigmergiagentic_validation() -> dict[str, Any]:
    runs: list[dict[str, Any]] = []
    query_dir = OUR_ROOT / 'queries'
    query_dir.mkdir(parents=True, exist_ok=True)

    for query_idx in range(MAX_QUERIES):
        cmd = [
            'uv', 'run', 'python', 'scripts/run_travelplanner_query_export.py',
            '--objective', f'Query {query_idx}',
            '--query-idx', str(query_idx),
            '--config', str(OUR_CONFIG_PATH),
            '--max-ticks', str(OUR_MAX_TICKS),
            '--agents', str(OUR_AGENT_COUNT),
            '--seed', '42',
        ]
        proc = run_command(
            cmd,
            cwd=REPO_ROOT,
            check=True,
            log_path=query_dir / f'query_{query_idx:03d}.log',
        )
        payload = extract_last_json(proc.stdout)
        (query_dir / f'query_{query_idx:03d}.json').write_text(
            json.dumps(payload, indent=2, ensure_ascii=False) + '\n',
            encoding='utf-8',
        )
        runs.append(payload)

    write_json(OUR_RUNS_JSON, {'runs': runs})
    run_command(
        [
            'uv', 'run', 'python', 'scripts/eval_travelplanner_official.py',
            '--runs-json', str(OUR_RUNS_JSON),
            '--database-root', 'data/travelplanner/database',
            '--split', SPLIT,
            '--out', str(OUR_OFFICIAL_JSON),
        ],
        cwd=REPO_ROOT,
        check=True,
        log_path=OUR_ROOT / 'official_eval.log',
    )
    official_payload = load_json(OUR_OFFICIAL_JSON, {})
    return official_payload.get('scores', {}) if isinstance(official_payload, dict) else {}


our_scores = None
if RUN_OUR:
    our_scores = run_stigmergiagentic_validation()
our_scores


## Arm 3: SwarmAgentic

This arm:
1. clones SwarmAgentic
2. patches only the provider/model compatibility layer for OpenRouter
3. trains with PSO on `qwen/qwen3.5-9b`
4. exports `save_state.jsonl -> save.jsonl`
5. runs `test.py`
6. converts results to the local `runs.json` format
7. evaluates with the same official scorer


In [ ]:
def clone_or_refresh_swarmagentic() -> None:
    if SWARM_CLONE.exists() and not CLONE_SWARM:
        print('Reusing existing clone:', SWARM_CLONE)
        return
    if SWARM_CLONE.exists():
        run_command(['rm', '-rf', str(SWARM_CLONE)], cwd=REPO_ROOT, check=True)
    run_command(
        ['git', 'clone', '--depth', '1', 'https://github.com/YaoZ720/SwarmAgenticCode.git', str(SWARM_CLONE)],
        cwd=REPO_ROOT,
        check=True,
        log_path=SWARM_ROOT / 'git_clone.log',
    )
    run_command(
        ['python', 'scripts/prepare_swarmagentic_openrouter.py', '--repo-root', str(SWARM_CLONE)],
        cwd=REPO_ROOT,
        check=True,
        log_path=SWARM_ROOT / 'patch_openrouter.log',
    )


def install_swarmagentic_deps() -> None:
    venv_python = SWARM_CLONE / '.venv_compare' / 'bin' / 'python'
    if INSTALL_SWARM_DEPS or not venv_python.exists():
        run_command(['uv', 'venv', '.venv_compare'], cwd=SWARM_CLONE, check=True, log_path=SWARM_ROOT / 'venv.log')
        run_command(
            ['uv', 'pip', 'install', '--python', str(venv_python), '-r', 'requirements.txt'],
            cwd=SWARM_CLONE,
            check=True,
            log_path=SWARM_ROOT / 'pip_install.log',
        )


def run_swarmagentic_validation() -> dict[str, Any]:
    clone_or_refresh_swarmagentic()
    install_swarmagentic_deps()
    swarm_cwd = SWARM_CLONE / 'travelplanner' / 'swarm'
    swarm_python = SWARM_CLONE / '.venv_compare' / 'bin' / 'python'
    swarm_env = {
        'OPENAI_API_KEY': os.environ['OPENROUTER_API_KEY'],
        'OPENAI_BASE_URL': OPENROUTER_BASE_URL,
    }

    if TRAIN_SWARM:
        run_command(
            [
                str(swarm_python), 'pso.py',
                '--max_iteration', str(SWARM_MAX_ITERATION),
                '--model', MODEL_NAME,
                '--max_workers', str(SWARM_MAX_WORKERS),
                '--sample_step', str(SWARM_SAMPLE_STEP),
                '--dataset', 'data/train_45.jsonl',
                '--ref_info', 'data/train_ref_info.jsonl',
                '--save_dir', str((SWARM_ROOT / 'training').resolve()),
            ],
            cwd=swarm_cwd,
            env=swarm_env,
            check=True,
            log_path=SWARM_ROOT / 'pso_train.log',
        )

    run_command(
        [
            'python', 'scripts/export_swarmagentic_save_jsonl.py',
            '--state-jsonl', str((swarm_cwd / 'save_state.jsonl').resolve()),
            '--out', str((swarm_cwd / 'save.jsonl').resolve()),
        ],
        cwd=REPO_ROOT,
        check=True,
        log_path=SWARM_ROOT / 'export_save_jsonl.log',
    )

    run_command(
        [
            str(swarm_python), 'test.py',
            '--particle_idx', '-1',
            '--model', MODEL_NAME,
            '--extract_model', SWARM_EXTRACT_MODEL,
            '--save_dir', str((SWARM_ROOT / 'evaluation' / 'test').resolve()),
            '--start_index', '0',
            '--end_index', str(MAX_QUERIES),
            '--max_workers', str(SWARM_MAX_WORKERS),
            '--dataset', 'data/validation.jsonl',
            '--ref_info', 'data/validation_ref_info.jsonl',
        ],
        cwd=swarm_cwd,
        env=swarm_env,
        check=True,
        log_path=SWARM_ROOT / 'test_eval.log',
    )

    run_command(
        [
            'python', 'scripts/convert_swarmagentic_travelplanner_results.py',
            '--results-jsonl', str(SWARM_RESULTS_JSONL),
            '--expected-count', str(MAX_QUERIES),
            '--out', str(SWARM_RUNS_JSON),
            '--method-name', 'SwarmAgentic',
        ],
        cwd=REPO_ROOT,
        check=True,
        log_path=SWARM_ROOT / 'convert_runs.log',
    )

    run_command(
        [
            'uv', 'run', 'python', 'scripts/eval_travelplanner_official.py',
            '--runs-json', str(SWARM_RUNS_JSON),
            '--database-root', 'data/travelplanner/database',
            '--split', SPLIT,
            '--out', str(SWARM_OFFICIAL_JSON),
        ],
        cwd=REPO_ROOT,
        check=True,
        log_path=SWARM_ROOT / 'official_eval.log',
    )
    official_payload = load_json(SWARM_OFFICIAL_JSON, {})
    return official_payload.get('scores', {}) if isinstance(official_payload, dict) else {}


swarm_scores = None
if RUN_SWARM:
    swarm_scores = run_swarmagentic_validation()
swarm_scores


## Final table

The next cell renders one final table with the three official scorer outputs.


In [ ]:
comparison_rows = []
if SOLO_OFFICIAL_JSON.exists():
    comparison_rows.append(f'Solo={SOLO_OFFICIAL_JSON}')
if OUR_OFFICIAL_JSON.exists():
    comparison_rows.append(f'StigmergiAgentic={OUR_OFFICIAL_JSON}')
if SWARM_OFFICIAL_JSON.exists():
    comparison_rows.append(f'SwarmAgentic={SWARM_OFFICIAL_JSON}')

if len(comparison_rows) < 3:
    raise RuntimeError('All three official eval JSON files must exist before rendering the final table.')

cmd = ['python', 'scripts/render_travelplanner_comparison_table.py']
for item in comparison_rows:
    cmd.extend(['--run', item])
cmd.extend([
    '--out-md', str(TABLE_MD),
    '--out-csv', str(TABLE_CSV),
    '--out-json', str(TABLE_JSON),
])
run_command(cmd, cwd=REPO_ROOT, check=True, log_path=TABLE_ROOT / 'render_table.log')

table_md = TABLE_MD.read_text(encoding='utf-8')
display(Markdown(table_md))

notebook_summary = {
    **summary,
    'solo_official_scores': load_json(SOLO_OFFICIAL_JSON, {}).get('scores', {}),
    'our_official_scores': load_json(OUR_OFFICIAL_JSON, {}).get('scores', {}),
    'swarm_official_scores': load_json(SWARM_OFFICIAL_JSON, {}).get('scores', {}),
    'table_md': str(TABLE_MD),
    'table_csv': str(TABLE_CSV),
    'table_json': str(TABLE_JSON),
}
write_json(NOTEBOOK_SUMMARY, notebook_summary)
NOTEBOOK_SUMMARY


## Suggested reporting sentence

> We compare three setups under the same OpenRouter-routed `qwen/qwen3.5-9b` model and the same official TravelPlanner scorer: a single-call solo planner, StigmergiAgentic V3, and SwarmAgentic. The comparison is controlled on provider, routed model, split, and scorer, while remaining framework-level rather than fully cost-matched because SwarmAgentic includes a PSO optimization phase before evaluation.
